# Model Selection — Income Investment

Per-model outputs, ablation study, and statistical comparison for the **IncomeInvestment** target.

**Prerequisites:** run all `utils/*.py` scripts first so that `data/pickled_files/` is populated.

**Source of truth:** `AT_comments.md`

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from scipy.stats import wilcoxon
from sklearn.metrics import ConfusionMatrixDisplay

sys.path.insert(0, str(Path().resolve()))
from utils.preprocessing import FEATURE_NAMES, RAW_FEATURE_NAMES, TARGETS

TARGET = 'IncomeInvestment'
PICKLE_ROOT = Path('data/pickled_files')
MODEL_KEYS = [
    'linear_reg', 'naive_bayes', 'rand_forest',
    'xgboost_shap', 'mlp', 'soft_voting_ens', 'hard_voting_ens',
]
# classifier_chain stores both targets in one file — handled separately below

## 1. Load pickled results

In [ ]:
results = {}
for key in MODEL_KEYS:
    path = PICKLE_ROOT / key / f'{TARGET.lower()}.joblib'
    if path.exists():
        results[key] = joblib.load(path)
        print(f'  Loaded: {key}')
    else:
        print(f'  MISSING: {key} — run python -m utils.{key} first')

# ClassifierChain: extract IncomeInvestment slice
chain_path = PICKLE_ROOT / 'classifier_chain' / 'both_targets.joblib'
if chain_path.exists():
    chain_full = joblib.load(chain_path)
    results['classifier_chain'] = {
        'model_name': chain_full['model_name'],
        'cv_metrics_raw': chain_full['cv_metrics_raw'][TARGET],
        'cv_metrics_summary': chain_full['cv_metrics_summary'][TARGET],
        'test_metrics': chain_full['test_metrics'][TARGET],
        'y_test_true': chain_full['y_test_true'][:, 0],
        'y_test_pred': chain_full['y_test_pred'][:, 0],
        'ablation': {
            'engineered': {
                'cv_metrics_raw': chain_full['ablation']['engineered']['cv_metrics_raw'][TARGET],
                'test_metrics': chain_full['ablation']['engineered']['test_metrics'][TARGET],
            },
            'raw': {
                'cv_metrics_raw': chain_full['ablation']['raw']['cv_metrics_raw'][TARGET],
                'test_metrics': chain_full['ablation']['raw']['test_metrics'][TARGET],
            },
        },
    }
    print('  Loaded: classifier_chain (Income slice)')

## 2. Test-set metrics summary

In [ ]:
rows = []
for key, r in results.items():
    tm = r['test_metrics']
    cv_summary = r.get('cv_metrics_summary', {})
    rows.append({
        'model': key,
        'accuracy':  round(tm['accuracy'], 3),
        'precision': round(tm['precision'], 3),
        'recall':    round(tm['recall'], 3),
        'f1':        round(tm['f1'], 3),
        'cv_f1_mean': round(cv_summary.get('f1', {}).get('mean', float('nan')), 3),
        'cv_f1_std':  round(cv_summary.get('f1', {}).get('std', float('nan')), 3),
    })

summary_df = pd.DataFrame(rows).set_index('model').sort_values('f1', ascending=False)
summary_df.style.highlight_max(subset=['f1', 'recall', 'precision'], color='lightgreen')

## 3. CV stability — F1 score distributions across folds

In [ ]:
cv_data = []
for key, r in results.items():
    for score in r['cv_metrics_raw']['f1']:
        cv_data.append({'model': key, 'cv_f1': score})

cv_df = pd.DataFrame(cv_data)

fig, ax = plt.subplots(figsize=(12, 5))
order = summary_df.index.tolist()
sns.boxplot(data=cv_df, x='model', y='cv_f1', order=order, ax=ax, palette='Set2')
ax.set_title(f'CV F1 Distributions — {TARGET}')
ax.set_xlabel('')
ax.set_ylabel('F1 Score (5-fold CV)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 4. Pairwise Wilcoxon signed-rank tests on CV F1

Tests whether the CV F1 score distributions of any two models differ significantly (p < 0.05).
With only 5 folds the test has low power; treat results as indicative, not conclusive.

In [ ]:
model_keys = list(results.keys())
cv_f1s = {k: results[k]['cv_metrics_raw']['f1'] for k in model_keys}

p_matrix = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
for m1 in model_keys:
    for m2 in model_keys:
        if m1 == m2:
            p_matrix.loc[m1, m2] = 1.0
        else:
            try:
                _, p = wilcoxon(cv_f1s[m1], cv_f1s[m2])
            except ValueError:
                p = 1.0  # identical distributions
            p_matrix.loc[m1, m2] = round(p, 3)

print(f'Wilcoxon p-values — {TARGET}')
p_matrix.style.background_gradient(cmap='RdYlGn_r', vmin=0, vmax=0.1)

## 5. Ablation study — engineered features vs raw baseline

Positive delta_F1 confirms that the interaction features add predictive value.

In [ ]:
abl_rows = []
for key, r in results.items():
    if 'ablation' not in r or r['ablation'] is None:
        continue
    eng_f1 = r['ablation']['engineered']['test_metrics']['f1']
    raw_f1 = r['ablation']['raw']['test_metrics']['f1']
    abl_rows.append({
        'model': key,
        'engineered_f1': round(eng_f1, 3),
        'raw_f1': round(raw_f1, 3),
        'delta_f1': round(eng_f1 - raw_f1, 3),
    })

abl_df = pd.DataFrame(abl_rows).set_index('model').sort_values('delta_f1', ascending=False)
abl_df.style.bar(subset=['delta_f1'], color=['#d65f5f', '#5fba7d'])

## 6. Best model selection

In [ ]:
best_key = summary_df['f1'].idxmax()
second_key = summary_df['f1'].iloc[1:].idxmax() if len(summary_df) > 1 else None

print(f'Best model:   {best_key}  (Test F1={summary_df.loc[best_key, "f1"]:.3f})')
if second_key:
    print(f'Second best:  {second_key}  (Test F1={summary_df.loc[second_key, "f1"]:.3f})')
    try:
        _, p_best = wilcoxon(cv_f1s[best_key], cv_f1s[second_key])
        sig = 'SIGNIFICANT' if p_best < 0.05 else 'not significant'
        print(f'Wilcoxon p={p_best:.3f} ({sig})')
    except ValueError:
        print('Wilcoxon: identical CV distributions (p=1.0)')

## 7. Confusion matrix — best model

In [ ]:
r_best = results[best_key]
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    r_best['y_test_true'],
    r_best['y_test_pred'],
    display_labels=['No', 'Yes'],
    ax=ax,
    colorbar=False,
)
ax.set_title(f'Confusion Matrix — {best_key} ({TARGET})')
plt.tight_layout()
plt.show()

## 8. SHAP global feature importances (XGBoost)

SHAP values are pre-computed and stored in the XGBoost pickle.

In [ ]:
xgb_path = PICKLE_ROOT / 'xgboost_shap' / f'{TARGET.lower()}.joblib'
if xgb_path.exists():
    xgb_result = joblib.load(xgb_path)
    shap_values = xgb_result['shap_values']
    shap_test_X = xgb_result['shap_test_X']

    print(f'SHAP values shape: {np.array(shap_values).shape}')
    plt.figure(figsize=(9, 5))
    shap.summary_plot(shap_values, shap_test_X, plot_type='bar', show=False)
    plt.title(f'SHAP Feature Importances — XGBoost ({TARGET})')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(9, 5))
    shap.summary_plot(shap_values, shap_test_X, show=False)
    plt.title(f'SHAP Beeswarm — XGBoost ({TARGET})')
    plt.tight_layout()
    plt.show()
else:
    print('XGBoost pickle not found — run python -m utils.xgboost_shap first')

## 9. Precision–Recall threshold (MiFID II compliance)

The operating threshold is chosen from the PR curve to maximise F1 subject to
**Precision ≥ 0.75** — the hard MiFID II constraint against automated mis-selling.

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay
import matplotlib.pyplot as plt

for key, r in results.items():
    thr_info = r.get('threshold_info')
    if thr_info is None:
        print(f'{key}: no threshold info stored')
        continue
    print(f'{key:30s}  threshold={thr_info["threshold"]:.3f}  '
          f'P={thr_info["precision"]:.3f}  R={thr_info["recall"]:.3f}  '
          f'F1={thr_info["f1"]:.3f}')

# Plot PR curve for best model
r_best = results[best_key]
if r_best.get('threshold_info'):
    ti = r_best['threshold_info']
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(ti['recalls'], ti['precisions'], lw=2, label=f'{best_key} PR curve')
    ax.axhline(0.75, color='red', linestyle='--', label='Precision floor (MiFID II = 0.75)')
    ax.scatter([ti['recall']], [ti['precision']], s=120, zorder=5,
               color='green', label=f'Selected threshold={ti["threshold"]:.3f}')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'Precision–Recall Curve — {best_key} ({TARGET})')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 10. Calibration — Brier scores & reliability diagrams

In [ ]:
from sklearn.calibration import calibration_curve

brier_rows = []
for key, r in results.items():
    bs = r.get('brier_score')
    bs_pre = r.get('brier_score_pre_cal')
    brier_rows.append({
        'model': key,
        'brier_pre_cal': round(bs_pre, 4) if bs_pre is not None else '—',
        'brier_post_cal': round(bs, 4) if bs is not None else '—',
    })

print(f'Brier scores — {TARGET}  (lower is better; 0.25 = no-skill baseline)')
pd.DataFrame(brier_rows).set_index('model')

In [ ]:
# Reliability diagrams for models with predict_proba
fig, axes = plt.subplots(1, min(4, len(results)), figsize=(4 * min(4, len(results)), 4))
if len(results) == 1:
    axes = [axes]

for ax, (key, r) in zip(axes, list(results.items())[:4]):
    path = PICKLE_ROOT / key.replace('classifier_chain', '') / f'{TARGET.lower()}.joblib'
    # reload to get model object
    pkl_path = PICKLE_ROOT / key / f'{TARGET.lower()}.joblib'
    if not pkl_path.exists():
        continue
    rr = joblib.load(pkl_path)
    model_obj = rr.get('model')
    if model_obj is None or not hasattr(model_obj, 'predict_proba'):
        ax.set_title(f'{key}\n(no predict_proba)')
        continue
    feat_names = rr.get('feature_names', FEATURE_NAMES)
    from utils.preprocessing import load_data, build_features, build_baseline_features, split_data, BASELINE_FEATURE_NAMES
    df_tmp = load_data()
    X_tmp = build_features(df_tmp) if feat_names == FEATURE_NAMES else build_baseline_features(df_tmp)
    _, X_te_tmp, _, y_te_tmp = split_data(X_tmp, df_tmp[TARGET])
    try:
        prob_true, prob_pred = calibration_curve(y_te_tmp, model_obj.predict_proba(X_te_tmp)[:, 1], n_bins=10)
        ax.plot(prob_pred, prob_true, marker='o', label=key)
        ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
        ax.set_title(key, fontsize=8)
        ax.set_xlabel('Mean predicted')
        ax.set_ylabel('Fraction positive')
    except Exception as e:
        ax.set_title(f'{key}\n({e})')

plt.suptitle(f'Reliability Diagrams — {TARGET}', y=1.02)
plt.tight_layout()
plt.show()

## 11. Label sensitivity analysis

Re-run the best model after randomly flipping 5% and 10% of training labels
(5 repetitions each) to verify robustness to the revealed-preference labelling
assumption (Section 3.1 of the paper). A degradation < 2 F1 points at 10%
corruption confirms the pipeline is label-robust.

In [ ]:
from utils.preprocessing import flip_labels, build_features, split_data, compute_test_metrics, load_data
import copy

def _sensitivity_run(target_col, flip_rate, n_repeats=5):
    df_s = load_data()
    y_s = df_s[target_col]
    X_s = build_features(df_s)
    X_tr_s, X_te_s, y_tr_s, y_te_s = split_data(X_s, y_s)
    f1_scores = []
    for rep in range(n_repeats):
        y_tr_noisy = flip_labels(y_tr_s, flip_rate, random_state=rep)
        # Use the best model's factory — reload and refit
        pkl_path = PICKLE_ROOT / best_key / f'{target_col.lower()}.joblib'
        if not pkl_path.exists():
            return None
        # Rebuild a fresh model from the stored class name
        rr = joblib.load(pkl_path)
        mdl = copy.deepcopy(rr['model'])
        try:
            mdl.fit(X_tr_s, y_tr_noisy)
            f1_scores.append(compute_test_metrics(mdl, X_te_s, y_te_s)['f1'])
        except Exception:
            pass
    return f1_scores

print(f'Label sensitivity — {TARGET} (best model: {best_key})')
print(f'{"Corruption":>12}  {"Mean F1":>8}  {"Std F1":>8}')
baseline_f1 = results[best_key]['test_metrics']['f1']
print(f'{"0% (baseline)":>12}  {baseline_f1:.3f}     —')
for rate in [0.05, 0.10]:
    scores = _sensitivity_run(TARGET, rate)
    if scores:
        print(f'{rate*100:>11.0f}%  {np.mean(scores):.3f}     {np.std(scores):.3f}')

## 12. Theoretical justification

### Why IncomeInvestment is harder to predict (38% prevalence, imbalanced)

IncomeInvestment clients are older, wealthier individuals who need regular cash flows — a relatively rare profile in the 5000-client dataset (38% positive). This imbalance biases classifiers toward the majority class, which is why **recall** is the primary optimisation criterion here: we penalise missed high-need clients more than false alarms.

### Lifecycle hypothesis

The `Age_x_Wealth` interaction term (Age × log-Wealth) encodes the financial lifecycle: the strongest IncomeInvestment signal comes from clients who are both elderly **and** wealthy. SHAP confirms this feature dominates predictions.

### Why two separate classifiers

Target correlation r ≈ 0.011 (documented in AT_comments.md). Multi-output classifiers (ClassifierChain) assume one target informs the other; with near-zero correlation this assumption fails. Two independent classifiers are therefore the correct modelling choice. The ClassifierChain result in this notebook quantifies how much (if any) lift the joint approach provides.

### Data leakage fix

The professor's baseline fit MinMaxScaler on the full dataset before splitting. This leaks test-set statistics into the training pipeline. All scripts in `utils/` fix this by splitting first, then fitting the scaler on the training set only.

## 13. Conclusions

In [ ]:
print(f'Best model for {TARGET}: {best_key}')
print()
print('Test metrics:')
for metric, val in results[best_key]['test_metrics'].items():
    print(f'  {metric:12s}: {val:.3f}')
print()

# Feature engineering lift summary
if best_key in abl_df.index:
    delta = abl_df.loc[best_key, 'delta_f1']
    direction = 'lift' if delta > 0 else 'drop'
    print(f'Feature engineering {direction} for best model: delta_F1 = {delta:+.3f}')